# Universidad Central · Big Data · Sesión 13
# ¿Cuándo basta pandas y cuándo necesitamos distribuir datos y modelos?

**Caso continuo:** Laura / Compras Claras. **Entorno principal:** Codespaces — elegir perfil **S13 · Dask, MapReduce y modelos SECOP**.

[Presentación y tutorial S13](../Presentaciones/s13-de-una-maquina-a-un-cluster.html#s1) · [Datos de referencia](../Datos/tc2_corpus_2026.json)

**No dependemos de que S12 ya se haya impartido.** El Taller 2 es una actividad evaluativa con su propia fecha. Aquí utilizamos dos fuentes versionadas:
- **200 expedientes** del Taller 2 para continuidad narrativa, *sin etiqueta de modalidad*.
- **1.000 registros SECOP** para entrenar un clasificador didáctico de modalidad, *no de fraude*.

Al final obtendrás datos Parquet, un manifiesto de origen, un clasificador evaluado y una explicación sobre cuándo Dask agrega valor. Todos los registros base son del repositorio; no necesitas APIs ni cuentas de pago.


## Mapa del laboratorio

1. Elegir herramienta: pandas, SQL/DuckDB, Dask, Spark.
2. Entender archivos distribuidos, bloques y réplicas sin instalar Hadoop.
3. Resolver MapReduce manualmente, luego en Python.
4. Entrar a Codespaces y comprobar bibliotecas.
5. Comparar pandas con Dask sobre el **mismo archivo SECOP**.
6. Entrenar **un modelo de scikit-learn** sobre modalidad.
7. Coordinar **varios entrenamientos independientes** con Dask Distributed.
8. Registrar un fallo controlado, guardar Parquet y adaptar el método.

**Convención:** después de cada salida encontrarás: **Cómo se lee / Qué nos dice / Qué NO permite concluir / Error común.** Los tiempos están sólo en el libreto docente, no aquí.


## 0 · Decidir antes de instalar

**Pregunta profesional:** ¿qué problema justifica pagar el costo de coordinar trabajadores?

| Trabajo | Primera opción razonable |
|---|---|
| 200 registros que caben en memoria | pandas |
| Consulta SQL sobre muchos CSV/Parquet | considerar DuckDB |
| Muchas transformaciones independientes o datos mayores que memoria | Dask puede ayudar |
| Un modelo clásico con cientos de ejemplos | scikit-learn |
| Un único modelo profundo entrenado entre máquinas | algoritmo con soporte distribuido explícito |

**OJO:** Dask no es sinónimo de «más rápido». Dos workers dentro de un Codespace **no son dos computadores**. Cada herramienta resuelve una necesidad distinta.


## 1 · Crear tu entorno propio en el navegador

**HAZ ESTO AHORA.** 1. Abre [el repositorio](https://github.com/jazaineam1/BigData2026). 2. Pulsa **Code → Codespaces → ⋯ → New with options**. 3. En configuración del contenedor selecciona **S13 · Dask, MapReduce y modelos SECOP**. **No** elijas «Airflow Lab» (predeterminado) ni «S11 multimedia». 4. Crea un Codespace y abre una **Terminal → New Terminal**.

Dentro de la terminal ejecuta:

```bash
python --version
python -m pip show pandas
python -m pip show dask
python -m pip show scikit-learn
python Laboratorios/S13_Codespaces/verificar_entorno.py
```

Si falta una biblioteca, **dentro de Codespaces**, no en tu PC: `python -m pip install -r Laboratorios/S13_Codespaces/requirements.txt`. Abre este cuaderno en VS Code web y selecciona el kernel Python del Codespace. No necesita Ubuntu, Docker ni VS Code instalados localmente.

**Resultados:** debe terminar con `LISTO PARA S13`; si escogiste el perfil equivocado, crea otro Codespace con la configuración S13.


### 1A · Preparar rutas (por qué no importa si abriste el cuaderno desde Cuadernos)

`Path` maneja rutas; `sys.path` indica dónde encontrar las funciones reutilizables. Conserva visible el código: puedes adaptarlo a un proyecto ajeno.


In [ ]:
from pathlib import Path
import sys

RAIZ = Path.cwd()
if not (RAIZ / "Datos").exists():
    RAIZ = RAIZ.parent

assert (RAIZ / "Datos/tc2_corpus_2026.json").exists(), "Abre el repositorio en Codespaces"
sys.path.insert(0, str(RAIZ / "Laboratorios/S13_Codespaces"))
print("Raíz del curso:", RAIZ)


### 1B · Comprobar biblioteca y archivos, no sólo que pip haya respondido

Esta celda ejecuta un cálculo mínimo Dask y comprueba que existen las dos fuentes del curso.


In [ ]:
import pandas as pd
import dask.dataframe as dd
from analisis import leer_corpus, leer_etiquetados

demo = pd.DataFrame({"grupo": ["A", "A", "B"], "valor": [10, 20, 30]})
mini = dd.from_pandas(demo, npartitions=2)
print(mini.groupby("grupo")["valor"].sum().compute())
assert int(mini.groupby("grupo")["valor"].sum().compute().loc["A"]) == 30
print("Fuentes y herramientas:", len(leer_corpus()), len(leer_etiquetados()))


**Cómo se lee:** A suma 30; B suma 30. Se encuentran 200 expedientes de referencia y 1.000 procesos etiquetados.  
**Qué nos dice:** pandas y Dask están disponibles y pueden ejecutar un cálculo.  
**Qué NO permite concluir:** que haya un clúster físico o mayor rendimiento que pandas.  
**Error común:** asumir que «pip instaló» equivale a «el flujo completo funciona».


## 2 · Sistemas de archivos: bloques, nodos y réplicas

Un sistema como **HDFS** divide archivos en bloques lógicos y almacena esos bloques en nodos. La replicación crea copias para tolerar fallos. El **NameNode** administra metadatos de ubicación; los **DataNodes** guardan bloques.

Esto NO es lo mismo que partir un DataFrame para procesarlo. La siguiente celda es una **SIMULACIÓN en Python**, no un clúster Hadoop real.

| Elemento | Función |
|---|---|
| Nodo | máquina o proceso que participa |
| Bloque | unidad de almacenamiento |
| Réplica | otra copia de un mismo bloque |
| Partición | subconjunto lógico de trabajo |
| Scheduler | coordinador de tareas |
| Worker | ejecutor de tareas |

**Decisión:** ¿qué sucede si se pierde un nodo que guarda la única copia de un bloque?


In [ ]:
ubicaciones = {
    "bloque_1": ["nodo_A", "nodo_B"],
    "bloque_2": ["nodo_B", "nodo_C"],
    "bloque_3": ["nodo_A", "nodo_C"],
}
nodo_caido = "nodo_B"

for bloque, nodos in ubicaciones.items():
    disponibles = [n for n in nodos if n != nodo_caido]
    print(bloque, "→", disponibles)
    assert disponibles, f"Bloque no recuperable: {bloque}"


**Cómo se lee:** cada bloque conserva una ubicación distinta de nodo_B.  
**Qué nos dice:** copiar bloques en distintos nodos puede proteger contra una falla simple.  
**Qué NO permite concluir:** que un Codespace sea un HDFS real ni que todas las políticas de replicación garanticen disponibilidad.  
**Error común:** confundir replicar los datos con paralelizar el cálculo.


## 3 · MapReduce: la trampa del promedio de promedios

Imagina 3 lotes de contrataciones con diferentes tamaños. En el primer lote hay 2 importes, en el segundo 5 y en el tercero 100. **Promediar las tres medias de los lotes es incorrecto** porque no representan el mismo número de observaciones.

`map` produce estadísticas parciales. `reduce` combina parciales sin perder los pesos. **Es el modelo MapReduce implementado en Python**, no un servicio de Apache Hadoop.


In [ ]:
lotes = [
    [100, 120],
    [10, 12, 11, 9, 8],
    [1] * 100,
]
parciales = [(sum(lote), len(lote)) for lote in lotes]
print("MAP (suma, cantidad):", parciales)


In [ ]:
suma_total = sum(suma for suma, cantidad in parciales)
cantidad_total = sum(cantidad for suma, cantidad in parciales)

promedio_correcto = suma_total / cantidad_total
promedio_incorrecto = sum(s / n for s, n in parciales) / len(parciales)

print("REDUCE correcto:", round(promedio_correcto, 3))
print("Promedio de promedios:", round(promedio_incorrecto, 3))
assert round(promedio_correcto, 6) == round(370/107, 6)


**Cómo se lee:** 370 unidades en 107 observaciones dan una media de aproximadamente 3,458.  
**Qué nos dice:** para un promedio distribuido necesitamos **suma y cantidad**, no sólo medias locales.  
**Qué NO permite concluir:** que ya usamos Hadoop MapReduce ni que un algoritmo cualquiera sea divisible así.  
**Error común:** `mean(mean(lote))` ignora los tamaños de las particiones.

**Transfiere:** esta estrategia sirve para métricas de ventas, llamadas o tiempos por cliente, siempre que conserves los pesos.


## 4 · ¿Qué trae el caso SECOP de Laura?

La S12 es el **Taller de Control 2**, no una clase de Dask terminada por definición. Su corpus de 200 expedientes no incluye la modalidad contractual como etiqueta para un modelo supervisado. Lo usamos sólo para recuperar la historia y comprobar identidad.


In [ ]:
referencia = leer_corpus()
print("Expedientes del taller:", len(referencia))
print(referencia[["id_proceso", "entidad"]].head(3))
assert referencia["id_proceso"].is_unique


### 4A · Datos para el problema supervisado

El CSV de SECOP versionado sí tiene `descripci_n_del_procedimiento` y `modalidad_de_contratacion`.

**Pregunta del modelo:** ¿podemos aprender a clasificar una modalidad registrada a partir del texto? **No es un detector de corrupción**.


In [ ]:
tabla = leer_etiquetados()
print("Dimensiones:", tabla.shape)
print(tabla["modalidad_de_contratacion"].value_counts().head(7))
print(tabla[["id_del_proceso", "descripci_n_del_procedimiento"]].head(2))
assert tabla["id_del_proceso"].is_unique


**Cómo se lee:** 1.000 filas, 59 columnas; «Contratación directa» y «Contratación régimen especial» son clases frecuentes.  
**Qué nos dice:** hay etiquetas registradas y texto para plantear una clasificación didáctica.  
**Qué NO permite concluir:** que esas frecuencias representen todo SECOP ni que el texto contenga evidencia de fraude.  
**Error común:** entrenar sobre los 200 expedientes del Taller 2 como si tuvieran esa etiqueta.


## 5 · pandas primero: la referencia de corrección

`groupby` agrupa filas por modalidad y `size()` cuenta filas. Esta salida es la referencia contra la cual compararemos Dask.


In [ ]:
conteo_pandas = tabla.groupby("modalidad_de_contratacion").size().sort_index()
print(conteo_pandas.head(10))
print("Total:", int(conteo_pandas.sum()))
assert conteo_pandas.sum() == 1000


## 6 · Dask DataFrame: el mismo cálculo por particiones

`from_pandas` divide el DataFrame en particiones. `groupby` construye operaciones diferidas; `compute()` materializa el resultado. Una partición **no** es un servidor.

¿Cuál sería el costo de Dask en sólo mil registros? El coordinador, el grafo de tareas y la comunicación pueden tardar más que la operación real.


In [ ]:
dask_tabla = dd.from_pandas(tabla, npartitions=4)
pendiente = dask_tabla.groupby("modalidad_de_contratacion").size()

print("Particiones:", dask_tabla.npartitions)
print("Antes de compute:", type(pendiente).__name__)
conteo_dask = pendiente.compute().sort_index()
valores_pandas = {str(k): int(v) for k, v in conteo_pandas.items()}
valores_dask = {str(k): int(v) for k, v in conteo_dask.items()}
assert valores_pandas == valores_dask
print("Resultado Dask:", conteo_dask.head(), "\nCoincide: SÍ")


### 6A · Medir sin inventar «Dask siempre gana»

Una medición debe incluir el tiempo de ejecución, el número de particiones y el tamaño del conjunto. Se repite en condiciones comparables; el primer tiempo puede incluir inicialización, cachés y procesos. Comparamos por **corrección** antes de velocidad.


In [ ]:
from analisis import comparar_pandas_dask
mediciones = []
for n_particiones in (1, 2, 4):
    mediciones.append(comparar_pandas_dask(tabla, npartitions=n_particiones))

display(pd.DataFrame(mediciones)[
    ["filas", "particiones", "segundos_pandas", "segundos_dask", "coinciden"]
])


**Cómo se lee:** los segundos se miden en TU Codespace y pueden cambiar en cada ejecución.  
**Qué nos dice:** la comparación es local, sobre el mismo conjunto de 1.000 registros.  
**Qué NO permite concluir:** cuál herramienta es más rápida para millones de filas en otro sistema.  
**Error común:** confundir aumentar particiones con aumentar máquinas o comparar ejecuciones con cargas distintas.


## 7 · Entrenamiento de modelos: ¿qué significa `fit()`?

**Entrenamiento**: el modelo ajusta parámetros usando ejemplos etiquetados. **Inferencia**: utiliza esos parámetros para clasificar nuevos textos.

Aquí `X` es una descripción de SECOP y `y` es su modalidad registrada. **TF-IDF** transforma palabras en variables numéricas. La regresión logística clasifica. La división entrenamiento/prueba se realiza **antes** de ajustar TF-IDF para evitar fuga de información estadística.

**Nunca entrenamos un modelo de «fraude» ni confundimos la etiqueta modalidad con irregularidad.**


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from analisis import preparar_clasificacion

muestra = preparar_clasificacion(tabla)
X = muestra["descripci_n_del_procedimiento"]
y = muestra["modalidad_de_contratacion"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print("Para aprender:", len(X_train), "| Para evaluar:", len(X_test))


In [ ]:
modelo = make_pipeline(
    TfidfVectorizer(max_features=1500, min_df=2),
    LogisticRegression(max_iter=500)
)
modelo.fit(X_train, y_train)             # aprende con textos de entrenamiento
prediccion = modelo.predict(X_test)      # infiere sobre textos no vistos
f1 = f1_score(y_test, prediccion, average="macro")
print("F1 macro observado:", round(f1, 4))


### 7A · Comparar con algo ingenuo

El clasificador de control siempre predice la clase mayoritaria. Si nuestro modelo no lo mejora, la complejidad no está justificada. F1 macro calcula F1 por clase y promedia sin ponderar por su frecuencia.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

control = DummyClassifier(strategy="most_frequent")
control.fit(X_train.to_frame(), y_train)
pred_control = control.predict(X_test.to_frame())
print("F1 macro control:", round(f1_score(y_test, pred_control, average="macro"), 4))
print("F1 macro modelo:", round(f1, 4))


**Cómo se lee:** F1 macro entre 0 y 1; comparar con el control usando el mismo test.  
**Qué nos dice:** rendimiento de un modelo sencillo para esta división de la muestra.  
**Qué NO permite concluir:** que funcione en otras entidades, periodos o modalidades; podría haber frases que revelan la etiqueta.  
**Error común:** ajustar TF-IDF antes de separar entrenamiento/prueba, o elegir parámetros observando reiteradamente el test.


## 8 · Dask Distributed: distribuir experimentos, no un único `fit`

Esta operación sí crea **un scheduler y dos workers lógicos** en Codespaces. Cada worker ejecuta una configuración independiente del modelo; dentro de cada tarea se usa scikit-learn normal.

**No es entrenamiento distribuido de un único modelo.** Para eso se necesita un algoritmo con soporte explícito, por ejemplo XGBoost Dask o Spark MLlib.


In [ ]:
from dask.distributed import Client
from sklearn.model_selection import cross_val_score

def probar(C, textos, etiquetas):
    candidato = make_pipeline(
        TfidfVectorizer(max_features=1500, min_df=2),
        LogisticRegression(C=C, max_iter=500)
    )
    resultados_cv = cross_val_score(
        candidato, textos, etiquetas, cv=3, scoring="f1_macro", n_jobs=1
    )
    return float(resultados_cv.mean())

with Client(processes=False, n_workers=2, threads_per_worker=1,
            dashboard_address=":8787") as cliente:
    print("Workers:", len(cliente.scheduler_info()["workers"]))
    print("Dashboard:", cliente.dashboard_link)
    valores_C = [0.1, 1.0, 10.0]
    textos_futuro = cliente.scatter(X_train, broadcast=True)
    etiquetas_futuro = cliente.scatter(y_train, broadcast=True)
    futuros = [
        cliente.submit(probar, C, textos_futuro, etiquetas_futuro, pure=False)
        for C in valores_C
    ]
    resultados_C = cliente.gather(futuros)

display(pd.DataFrame({"C": valores_C, "F1 CV": resultados_C}))


### 8A · ¿Para qué sirve el tablero Dask?

En Codespaces abre la pestaña **PORTS**, localiza **8787** y verifica que su visibilidad sea **Private**; abre la URL privada. En la vista de tareas observarás etapas de procesamiento y trabajadores. Si el puerto no se muestra, el resultado textual de `scheduler_info()` es suficiente.

**Cómo se lee:** cada tarea es un entrenamiento independiente.  
**Qué nos dice:** hay coordinación real con Dask dentro del Codespace.  
**Qué NO permite concluir:** que existan dos máquinas físicas o que el algoritmo `LogisticRegression.fit` esté distribuido.  
**Error común:** hablar de «entrenamiento distribuido» cuando sólo se paralelizan experimentos.


## 9 · Un error real, pero controlado

No vamos a matar un servidor ni borrar datos. Una tarea recibe entrada inválida, genera una excepción y después se ejecuta otra tarea correcta. Esto enseña diferencia entre **falla de tarea** y **caída de worker**.


In [ ]:
def duplicar(numero):
    if numero < 0:
        raise ValueError("Número inválido para el ejercicio")
    return numero * 2

with Client(processes=False, n_workers=2, threads_per_worker=1,
            dashboard_address=":8787") as cliente:
    try:
        cliente.submit(duplicar, -1, pure=False).result()
    except ValueError as error:
        print("Falló como esperábamos:", error)
    correcto = cliente.submit(duplicar, 4, pure=False).result()

print("Segundo intento controlado:", correcto)
assert correcto == 8


**Cómo se lee:** una tarea falla con entrada negativa y otra termina correctamente.  
**Qué nos dice:** podemos capturar fallos y decidir qué reintentar.  
**Qué NO permite concluir:** que Dask recupere automáticamente cualquier error lógico o garantice exactamente una escritura externa.  
**Error común:** confundir reintentar una tarea fallida con idempotencia de todo el sistema.


## 10 · Guardar evidencia reutilizable con Parquet

Parquet es un formato **columnar**. El archivo siguiente mantiene procesos únicos, modalidad y descripción; no guarda por sí mismo los pesos de un modelo entrenado. La sesión 14 lo leerá con PySpark.


In [ ]:
from analisis import escribir_resultados
manifest = escribir_resultados(tabla)
print("Salida:", manifest["salida"])
print("Filas:", manifest["filas"])
print("SHA-256 fuente:", manifest["sha256_fuente"][:20], "...")

archivo = RAIZ / "Laboratorios/S13_Codespaces/salidas/secop_s13.parquet"
reabierto = pd.read_parquet(archivo)
assert reabierto["id_del_proceso"].is_unique
print("Reabierto:", reabierto.shape)


**Cómo se lee:** se exportaron 1.000 procesos únicos de la muestra y se recuperaron desde Parquet.  
**Qué nos dice:** la entrega es tabular, trazable a una fuente específica y legible.  
**Qué NO permite concluir:** que el archivo Parquet tenga los modelos entrenados o preserve una licencia externa no registrada.  
**Error común:** confundir igualdad lógica de filas con igualdad byte a byte de dos archivos Parquet reescritos.


## 11 · Transferencia: convierte la técnica en tu propio proyecto

No memorices rutas del curso. En un proyecto nuevo identifica:

- **Archivo de entrada**: CSV o Parquet que controles y tengas permiso de usar.
- **Clave única**: ID que representa una observación.
- **Columna de agrupación**: qué necesitas comparar y por qué.
- **Etiqueta**: una variable que **exista realmente** si vas a entrenar supervisadamente.
- **Texto o variables**: de dónde provienen; ¿alguna revela directamente la etiqueta?
- **Control simple**: pandas antes de Dask.
- **Salida**: Parquet + manifiesto con fuente, conteos y decisiones.

En `Laboratorios/S13_Codespaces/analisis.py` puedes reemplazar `CSV`, `COL_TEXTO`, `COL_LABEL` y `CLASES` después de inspeccionar el nuevo esquema. Si la columna no existe, debes recibir un error visible, no fabricar valores.


## 12 · Autocomprobación técnica

Elige una herramienta para cada tarea **después de observar los resultados**, no sólo por definición:

| Caso | Primera respuesta defendible |
|---|---|
| 200 filas para un conteo | pandas |
| SQL sobre archivos grandes | considerar DuckDB |
| Varias transformaciones y tareas independientes | Dask según volumen y medición |
| Un modelo clásico pequeño | scikit-learn |
| Muchos entrenamientos de parámetros independientes | Dask Futures si la carga justifica coordinación |
| Un único modelo aprendido entre nodos | estimador con soporte distribuido explícito |

**Para llevar:** *procesar datos por particiones* ≠ *ejecutar varios modelos* ≠ *entrenar un modelo único de forma distribuida*.


## Cierre · Puente a la sesión 14

El resultado de la sesión 13 es `secop_s13.parquet`, un CSV fuente identificado, un modelo clásico evaluado y evidencias de coordinación. **En S14 usaremos PySpark para leer el mismo conjunto y comparar un tercer motor de procesamiento**, no volver a empezar otro caso.

**Errores comunes que evitamos:** usar Dask para 200 filas sin justificarlo; contar modalidades como irregularidades; promediar promedios; llamar «clúster físico» a dos workers locales; ajustar TF-IDF antes del corte; confundir inferencia y entrenamiento; comparar tiempos sin mismo trabajo.

**Fuentes oficiales:** [Dask](https://docs.dask.org/en/stable/) · [Dask Distributed](https://distributed.dask.org/) · [scikit-learn](https://scikit-learn.org/stable/) · [Codespaces](https://docs.github.com/es/codespaces).


## Extensión: 26.000 procesos SECOP y un único XGBoost distribuido

**Caso Compras Claras:** Laura compara el precio base registrado con un presupuesto de referencia estimado por ML. Se marcan discrepancias estadísticas **para revisión humana**, no fraude ni irregularidades.

**¿Por qué cambiamos de modelo?** En la sección anterior, Dask repartía entrenamientos independientes de scikit-learn. Ahora distribuiremos **las filas que necesita un único XGBoost** mediante `xgboost.dask.DaskDMatrix` y `xgboost.dask.train()`.

La fuente ya está en el repositorio: tres CSV de procesos SECOP II. Se seleccionan **26.000 fichas reales con ID único**, se entrenan **20.800** y se evalúan **5.200** más recientes. *No son 26.000 archivos PDF*.

[Código del motor editable](../Laboratorios/S13_Codespaces/modelo_distribuido_20k.py) · [Guía técnica de instalación, datos y métricas](../Laboratorios/S13_Codespaces/GUIA_MODELO_20K.md).


### 1 · Elegir variables antes de elegir modelos

| Sistema | Entradas | ¿Qué aprendizaje hay? |
|---|---|---|
| Control | mediana de precios de entrenamiento | no aprende relaciones |
| XGBoost A | modalidad, región, duración, tipo, categoría, mes | un modelo entrenado entre workers Dask |
| XGBoost B | igual + 16 ejes TF-IDF/SVD del texto | otro modelo distribuido para medir aporte del texto |
| EmbeddingGemma | embeddings neuronales del texto | **opcional futuro**, sólo si el costo y la mejora lo justifican |

**TF-IDF/SVD no son embeddings neuronales.** La variable objetivo es `log1p(precio_base)`; el logaritmo ayuda con montos de muy distinta escala.

**Fuga de información:** nunca entrenamos con `valor_total_adjudicacion`, `adjudicado` o respuestas posteriores. El texto podría mencionar precios o condiciones muy específicas; se debe examinar antes de desplegar.


### 2 · Código fundamental que debes entender

```python
dX = da.from_array(X_train, chunks=(2500, X_train.shape[1]))
dy = da.from_array(y_train, chunks=(2500,))
dtrain = xgboost.dask.DaskDMatrix(cliente, dX, dy, feature_names=nombres)
resultado = xgboost.dask.train(
    cliente,
    {"objective": "reg:squarederror", "tree_method": "hist"},
    dtrain, num_boost_round=30,
)
```

**Qué hace:** divide filas de entrenamiento en particiones, reparte los datos y coordina el cálculo de los árboles. El estimador XGBoost combina información entre workers para **un único modelo**, no sólo tres funciones `fit()` independientes.

**Dónde corre:** 2 workers con procesos separados en un Codespace remoto. Sigue siendo **una máquina física**. Un modelo local de 20.800 procesos podría ser más rápido; aquí investigamos cómo funciona el mecanismo distribuido.


### 3 · Ejecutar un entrenamiento real desde tu Codespace

El propio programa `modelo_distribuido_20k.py` permite leer y cambiar cada función, y expone los parámetros de tamaño, rondas y número de trabajadores. Prueba en la terminal:

```bash
python -m pip show xgboost
python Laboratorios/S13_Codespaces/modelo_distribuido_20k.py --registros 26000 --rondas 30 --workers 2
```

Si falta XGBoost, dentro del Codespace: `python -m pip install -r Laboratorios/S13_Codespaces/requirements.txt`.

Activa la celda siguiente poniendo `ACTIVAR_MODELO_26K=True`. Está desactivada por defecto para que **Run All del laboratorio básico** no repita una segunda ejecución costosa; la prueba independiente del proyecto sí realiza un entrenamiento real sobre los 26.000 procesos.


In [ ]:
import subprocess
import sys

ACTIVAR_MODELO_26K = False  # Cambiar a True para el entrenamiento real.
motor_ml = RAIZ / "Laboratorios/S13_Codespaces/modelo_distribuido_20k.py"
assert motor_ml.exists(), "Falta el motor de aprendizaje del repositorio"

if ACTIVAR_MODELO_26K:
    subprocess.run(
        [sys.executable, str(motor_ml), "--registros", "26000", "--rondas", "30"],
        cwd=RAIZ, check=True
    )
else:
    print("Extensión lista: pon ACTIVAR_MODELO_26K=True cuando quieras entrenar.")


### 4 · Ver resultados que existen de verdad

El entrenamiento produce un manifiesto de fuentes (SHA-256), dos modelos JSON, un Parquet con los 26.000 procesos, y predicciones sólo sobre la prueba temporal. Compara cada modelo con la **mediana del entrenamiento** usando el mismo periodo de prueba.

- **MAE(log)**: error en la escala logarítmica; menor es mejor.
- **Factor mediano**: diferencia multiplicativa típica; 1 es perfecto.
- **Dentro de ×2**: porcentaje de casos cuyo precio estimado está entre la mitad y el doble del observado.
- **Discrepancia ≥×3**: filtro estadístico de revisión, no acusación.

Para comprobar los resultados ejecutados desde la terminal, usa la siguiente celda.


In [ ]:
import json
from pathlib import Path
import pandas as pd

dir_modelo = RAIZ / "Laboratorios/S13_Codespaces/salidas/modelo_20k"
archivo_manifest = dir_modelo / "manifiesto.json"
if archivo_manifest.is_file():
    metadatos = json.loads(archivo_manifest.read_text(encoding="utf-8"))
    print("Fichas:", metadatos["observaciones_unicas"],
          "| entrenamiento:", metadatos["entrenamiento"],
          "| test:", metadatos["prueba"],
          "| workers:", metadatos["workers"])
    display(pd.DataFrame(metadatos["metricas"]).T)
else:
    print("No hay métricas 26K todavía: ejecuta el motor antes de interpretarlas.")


### 5 · Laboratorio de investigación visual

Después de ejecutar el motor, abre un panel interactivo en la **terminal del Codespace**:

```bash
streamlit run Laboratorios/S13_Codespaces/app_modelo_20k.py --server.address 0.0.0.0 --server.port 8502
```

Desde **PORTS**, abre el puerto **8502** con visibilidad **PRIVATE**. Filtra los procesos por modalidad y magnitud de discrepancia, mira la nube de precio real frente a previsto y abre la URL SECOP original para comprobar un caso.

**Cómo se lee:** compara factores y tres métricas entre modelos. **Qué nos dice:** dónde el modelo se equivoca más. **Qué NO permite concluir:** presupuesto justo, ilegalidad, daño fiscal o desviación causal. **Error común:** confundir una predicción con una auditoría jurídica.

**Puente a S14:** el archivo `muestra_26k.parquet` se puede leer con PySpark para comparar cómo cambiaría la ejecución en otro motor.
